# Task: 2.2: Classical Machine Learning

### Classification

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, confusion_matrix

### STEP 1: PREPARATION & CLEANING

In [22]:
df = pd.read_csv('data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [23]:
df.drop('Unnamed: 32', axis=1, inplace=True)

# FEATURE ENGINEERING
# Feature 1: Ratio of radius to texture to measure overall size relative to variation
df['radius_texture_ratio'] = df['radius_mean'] / (df['texture_mean'] + 1e-5)
# Feature 2: Overall severity interaction (product of area and concavity)
df['area_concavity_product'] = df['area_mean'] * df['concavity_mean']

# Correct the feature trap by dropping the metadata 'id' column
X = df.drop(columns=['id', 'diagnosis'])
y = LabelEncoder().fit_transform(df['diagnosis'])

# 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale data correctly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### STEP 2: ESTABLISH DUMB BASELINE (MANDATORY)

In [24]:
# Always predicts the majority class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)
base_pred = baseline.predict(X_test_scaled)

print("Dumb Baseline Metrics")
print(f"Accuracy: {accuracy_score(y_test, base_pred):.4f}\n")

Dumb Baseline Metrics
Accuracy: 0.6228



### STEP 3: TRAIN & COMPARE TWO+ ALGORITHMS

In [25]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

print(" Algorithm Comparison Table ")
comparison_data = []

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    
    # Track cross validation on 5 folds to ensure robust evaluations
    cv_score = cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='accuracy').mean()
    
    comparison_data.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "5-Fold CV Accuracy": cv_score
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

 Algorithm Comparison Table 
              Model  Test Accuracy  Precision   Recall  F1-Score  5-Fold CV Accuracy
Logistic Regression       0.982456   0.976744 0.976744  0.976744            0.971429
                KNN       0.956140   0.952381 0.930233  0.941176            0.960440
      Decision Tree       0.938596   0.928571 0.906977  0.917647            0.925275
      Random Forest       0.964912   0.975610 0.930233  0.952381            0.960440


### STEP 4: IMPROVE & VALIDATE (HYPERPARAMETER TUNING)

In [26]:
print("Tuning Best Performer")
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
}

grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
tuned_preds = best_model.predict(X_test_scaled)
tuned_cv = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy').mean()

Tuning Best Performer


### STEP 5: EXPERIMENT LOG

In [27]:
print("Final Experiment Log")
experiment_log = pd.DataFrame([
    {"Experiment": "Dumb Baseline", "CV Score / Baseline Score": accuracy_score(y_test, base_pred)},
    {"Experiment": "Initial Logistic Regression", "CV Score / Baseline Score": comparison_data[0]["5-Fold CV Accuracy"]},
    {"Experiment": "Tuned Logistic Regression (GridSearch)", "CV Score / Baseline Score": tuned_cv}
])
print(experiment_log.to_string(index=False))

Final Experiment Log
                            Experiment  CV Score / Baseline Score
                         Dumb Baseline                   0.622807
           Initial Logistic Regression                   0.971429
Tuned Logistic Regression (GridSearch)                   0.973626


In [28]:
print("\nTuned Confusion Matrix:")
print(confusion_matrix(y_test, tuned_preds))


Tuned Confusion Matrix:
[[71  0]
 [ 2 41]]
